# Diarka QAOA Portfolio — Notebook 03: First QAOA Run

**Week 2, Session 3.** Goal: run the actual QAOA algorithm on the Ising Hamiltonian built in Session 2, on a noiseless simulator, at depths $p = 1, 2, 3$. For each depth we measure:

1. **Approximation ratio** $\;r = (E_\max - \langle H\rangle) / (E_\max - E_\min)$ — how close the optimised expectation value comes to the true ground-state energy.
2. **Ground-state probability** — fraction of measurement shots that land exactly on the optimal portfolio bitstring.
3. **Best sampled energy** — the energy of the lowest-energy bitstring actually seen during sampling, which is the practically useful answer (you don't get $\langle H\rangle$ from hardware, you get bitstrings).

The end-state we're after: a clear demonstration that increasing $p$ improves the approximation, plus an honest accounting of QAOA's behaviour at modest depths on this specific Hamiltonian.

This is the first session where the *quantum* part of the project actually appears: a parameterised quantum circuit, a hybrid quantum-classical optimisation loop, and measurements.

## 0. Setup and load Session 2 outputs

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import warnings

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

# Silence the SparseEfficiencyWarning chatter from scipy that some Qiskit
# evolution-gate paths trigger. Doesn't affect results.
warnings.filterwarnings("ignore", category=Warning, module="scipy")

from qiskit.quantum_info import SparsePauliOp
from src.qaoa import (
    qaoa_ansatz,
    default_initial_parameters,
    optimise_qaoa,
    sample,
    approximation_ratio,
    ground_state_probability,
    top_k_bitstrings,
    bitstring_energy,
)

DATA_PROCESSED = ROOT / "data" / "processed"

# Reconstruct the Hamiltonian saved by Session 2.
ham_data = np.load(DATA_PROCESSED / "hamiltonian.npz", allow_pickle=False)

pauli_labels  = ham_data["pauli_labels"]
pauli_coeffs  = ham_data["pauli_coeffs"]
ISING_OFFSET  = float(ham_data["ising_offset"])
E_GROUND      = float(ham_data["ground_energy"])
GROUND_BITS   = str(ham_data["ground_bitstring"])
GROUND_SELECT = ham_data["ground_selection"].astype(int)
ENERGIES      = ham_data["sorted_energies"]
BITSTRINGS    = ham_data["sorted_bitstrings"]
N_QUBITS      = int(ham_data["n_qubits"])
TICKERS       = tuple(str(t) for t in ham_data["tickers"])

E_MAX = float(ENERGIES[-1])
SPECTRAL_GAP = float(ENERGIES[1] - ENERGIES[0])

H = SparsePauliOp.from_list(list(zip(pauli_labels, pauli_coeffs)))

print(f"Hamiltonian reconstructed.")
print(f"  qubits           = {N_QUBITS}")
print(f"  Pauli terms      = {len(H)}")
print(f"  E_min (target)   = {E_GROUND:+.6f}")
print(f"  E_max            = {E_MAX:+.6f}")
print(f"  Spectral gap     = {SPECTRAL_GAP:.6f}")
print(f"  Ground bitstring = {GROUND_BITS}")
print(f"  Ground selection = {GROUND_SELECT.tolist()}  ({', '.join(t for t, b in zip(TICKERS, GROUND_SELECT) if b)})")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## 1. The QAOA ansatz

QAOA prepares a parameterised quantum state

$$
|\psi(\boldsymbol\gamma, \boldsymbol\beta)\rangle \;=\; \prod_{k=1}^{p} U_M(\beta_k)\,U_C(\gamma_k)\; \cdot\; H^{\otimes n}|0\rangle^{\otimes n}
$$

where

$$
U_C(\gamma) = e^{-i\gamma \hat H_{\text{cost}}}, \qquad U_M(\beta) = e^{-i\beta \sum_i X_i}
$$

The cost layer $U_C$ shifts the relative phases of computational-basis states according to the Ising energies — low-energy states get less phase rotation than high-energy ones. The mixer layer $U_M$ then redistributes amplitude between bitstrings that differ by single-bit flips. Stacking $p$ such layers and tuning $(\boldsymbol\gamma, \boldsymbol\beta)$ steers the wavefunction toward low-energy bitstrings.

The classical optimisation problem is:

$$
\min_{\boldsymbol\gamma, \boldsymbol\beta} \; \langle\psi(\boldsymbol\gamma, \boldsymbol\beta) | \hat H |\psi(\boldsymbol\gamma, \boldsymbol\beta)\rangle
$$

In the noiseless simulator setting we use here, expectation values are computed exactly via statevector simulation. Once we move to real hardware, $\langle H\rangle$ would be estimated from finite-shot measurements with finite precision.

### Visualising the circuit at $p = 1$

In [3]:
ansatz_p1 = qaoa_ansatz(H, p=1)
print(f"Ansatz parameters: {ansatz_p1.num_parameters}  (2 × p = 2 × 1)")
print(f"Circuit depth:     {ansatz_p1.depth()}")
print()
# Decompose one level so we can see the cost-evolution gate's structure.
ansatz_p1.decompose(reps=1).draw("mpl", style="iqp", fold=80)

Ansatz parameters: 2  (2 × p = 2 × 1)
Circuit depth:     3



MissingOptionalLibraryError: "The 'pylatexenc' library is required to use 'MatplotlibDrawer'. You can install it with 'pip install pylatexenc'."

## 2. Optimise at $p = 1$

We use COBYLA as the classical optimiser — it's gradient-free, robust on noisy-looking cost surfaces, and is the standard choice in QAOA tutorials.

A **linear-ramp** initialisation is used: $\gamma$ ramps from $0.1 \to 0.5$, $\beta$ from $0.5 \to 0.1$. This empirically outperforms zero or random starts on most Ising landscapes.

Each step in the trace below corresponds to one expectation-value evaluation. COBYLA evaluates the function many times per "iteration" while updating its trust region, so the trace is longer than the formal iteration count.

In [ ]:
result_p1 = optimise_qaoa(H, p=1, maxiter=200)

print(f"p=1 optimisation")
print(f"  optimal ⟨H⟩       = {result_p1.optimal_energy:+.6f}")
print(f"  ground energy     = {E_GROUND:+.6f}")
print(f"  gap from ground   = {result_p1.optimal_energy - E_GROUND:.6f}")
print(f"  expectation evals = {len(result_p1.trace)}")
print(f"  scipy converged   = {result_p1.converged}")
print()
print(f"  optimal γ = {result_p1.optimal_params[0]:+.4f}")
print(f"  optimal β = {result_p1.optimal_params[1]:+.4f}")

## 3. Optimise at $p = 2$ and $p = 3$

In [ ]:
result_p2 = optimise_qaoa(H, p=2, maxiter=300)
print(f"p=2  ⟨H⟩ = {result_p2.optimal_energy:+.6f}  ({len(result_p2.trace)} evals)")

result_p3 = optimise_qaoa(H, p=3, maxiter=500)
print(f"p=3  ⟨H⟩ = {result_p3.optimal_energy:+.6f}  ({len(result_p3.trace)} evals)")

results = {1: result_p1, 2: result_p2, 3: result_p3}

## 4. Convergence behaviour

The classical optimisation trace at each depth, with the ground-state energy marked. Lower is better. Note that COBYLA's trust-region updates produce a non-monotone trace — what matters is the lowest point reached, not every step.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colors = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}
for p, res in results.items():
    ax.plot(res.trace, color=colors[p], alpha=0.7, label=f"p = {p}  (final ⟨H⟩ = {res.optimal_energy:+.4f})")

ax.axhline(E_GROUND, color="#d62728", linestyle="--", alpha=0.6,
           label=f"ground-state energy E_min = {E_GROUND:+.4f}")
ax.set_xlabel("Expectation-value evaluation")
ax.set_ylabel("⟨H⟩")
ax.set_title("QAOA classical-optimisation trace")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## 5. Sampling the optimised circuits

The optimiser found good parameters. To get a *bitstring* (the practically useful output), we sample the optimised circuit. With ``shots = 4096`` and an exact statevector simulator, the resulting counts give a faithful empirical distribution of $|\psi|^2$.

Each sample is a candidate portfolio. The most-frequently-sampled bitstrings should concentrate on low Ising energies.

In [ ]:
SHOTS = 4096
SEED  = 1234

for p in [1, 2, 3]:
    results[p] = sample(results[p], shots=SHOTS, seed=SEED)

# Build a tidy summary of approximation quality and sampling concentration.
print(f"{'p':>3}  {'⟨H⟩':>9}  {'approx ratio':>13}  {'best sampled E':>16}  {'P(ground)':>11}  {'unique bitstrings':>18}")
print("-" * 78)

summary = {}
for p in [1, 2, 3]:
    res = results[p]
    # Best sampled energy: minimum Ising energy across the bitstrings that
    # actually appeared in our shots.
    sampled_energies = [
        bitstring_energy(bits, ENERGIES, BITSTRINGS) for bits in res.counts
    ]
    best_sampled = min(sampled_energies)
    r = approximation_ratio(res.optimal_energy, ground_energy=E_GROUND, max_energy=E_MAX)
    gs_prob = ground_state_probability(res.counts, GROUND_BITS, shots=SHOTS)
    summary[p] = {
        "energy": res.optimal_energy,
        "ratio":  r,
        "best_sampled": best_sampled,
        "gs_prob": gs_prob,
        "unique": len(res.counts),
    }
    print(f"{p:>3}  {res.optimal_energy:>+9.4f}  {r:>13.4f}  {best_sampled:>+16.4f}  {gs_prob:>11.3%}  {len(res.counts):>18}")

print()
print(f"Ground-state energy reference: {E_GROUND:+.4f}")

### Interpretation: approximation ratio vs ground-state probability

Two metrics, telling different stories:

- **Approximation ratio** ($r$) measures how close $\langle H\rangle$ is to the ground-state energy. It's a smooth, averaged quantity.
- **Ground-state probability** measures how often the *exact* optimum appears in the bitstring sample.

These can decouple sharply when the Ising spectrum has many low-energy states close to the ground state. In that case QAOA correctly concentrates probability on the low-energy *region* (so $r$ is high) but the mass is spread across many near-degenerate competitors, so any single one — including the true optimum — has low probability.

For this particular portfolio Hamiltonian the spectral gap is 0.0000; you can see in the table above whether QAOA happens to concentrate or smear.

The **best sampled energy** is in some ways the most useful number — it's what you would get on real hardware: run the circuit, look at the resulting bitstrings, pick the best one.

## 6. Where is the sampled mass landing?

A side-by-side view: for each $p$, plot every sampled bitstring at its true Ising energy on the x-axis, with sample probability on the y-axis. Overlay the full Ising spectrum as tick marks at the top so you can see how dense the eigenvalue distribution is in each region.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, p in zip(axes, [1, 2, 3]):
    res = results[p]
    energies = np.array([
        bitstring_energy(bits, ENERGIES, BITSTRINGS) for bits in res.counts
    ])
    probs = np.array([c / SHOTS for c in res.counts.values()])

    # Sampled bitstrings as vertical stems at their energies.
    ax.vlines(energies, 0, probs, color="#1f77b4", alpha=0.5, lw=1)
    ax.scatter(energies, probs, s=12, color="#1f77b4", alpha=0.6)

    # Ground state highlighted.
    if GROUND_BITS in res.counts:
        ax.scatter([E_GROUND], [res.counts[GROUND_BITS] / SHOTS],
                   color="#d62728", s=90, zorder=5, marker="*",
                   label=f"ground state ({res.counts[GROUND_BITS]/SHOTS:.1%})")
    else:
        ax.axvline(E_GROUND, color="#d62728", linestyle="--", alpha=0.5,
                   label="ground state (not sampled)")

    # Mark all eigenvalues as small ticks at the top of the panel.
    yt = ax.get_ylim()[1]
    ax.scatter(ENERGIES, np.full_like(ENERGIES, yt * 0.95), marker="|",
               color="#aaaaaa", s=80, alpha=0.5)

    ax.set_ylabel("Sample probability")
    ax.set_title(f"p = {p}   |   ⟨H⟩ = {res.optimal_energy:+.4f}   |   approx ratio = {summary[p]['ratio']:.3f}")
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Ising energy of sampled bitstring  (grey ticks = full spectrum)")
plt.tight_layout()
plt.show()

## 7. Top-sampled portfolios

The most-frequently-sampled bitstrings, translated back to portfolio selections. For each, we show:

- the cardinality $\sum x_i$ (feasibility — should be 4),
- the assets selected,
- the Ising energy,
- the original portfolio objective $q\,x^{\!\top}\!\Sigma x - \mu^{\!\top} x$ (Ising energy + offset),
- the sample probability.

In [ ]:
for p in [1, 2, 3]:
    res = results[p]
    print(f"\n========== p = {p}: top 5 sampled portfolios ==========")
    print(f"{'bitstring':>10}  {'card':>4}  {'energy':>9}  {'portfolio obj':>14}  {'prob':>7}  selected assets")
    print("-" * 95)
    for bits, count in top_k_bitstrings(res.counts, 5):
        x = np.array([int(c) for c in reversed(bits)], dtype=int)
        cardinality = int(x.sum())
        energy = bitstring_energy(bits, ENERGIES, BITSTRINGS)
        obj    = energy + ISING_OFFSET
        prob   = count / SHOTS
        picks  = ", ".join(t for t, b in zip(TICKERS, x) if b) if cardinality > 0 else "(none)"
        mark   = "  ← ground state" if bits == GROUND_BITS else ""
        print(f"{bits:>10}  {cardinality:>4}  {energy:>+9.4f}  {obj:>+14.4f}  {prob:>6.1%}  {picks}{mark}")

## 8. Approximation-ratio comparison

The headline plot: approximation ratio improves (or saturates) with $p$. With only one classical seed per depth and a small problem, treat this as illustrative rather than statistically definitive — real reporting would average over multiple random restarts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ps = [1, 2, 3]
ratios = [summary[p]["ratio"] for p in ps]
gs_probs = [summary[p]["gs_prob"] for p in ps]

bars = ax.bar([p - 0.2 for p in ps], ratios, width=0.4, color="#1f77b4",
              label="Approximation ratio  (E_max − ⟨H⟩) / (E_max − E_min)")
bars2 = ax.bar([p + 0.2 for p in ps], gs_probs, width=0.4, color="#d62728",
               label="P(ground state) per shot")

ax.set_xticks(ps)
ax.set_xticklabels([f"p = {p}" for p in ps])
ax.set_ylabel("Value")
ax.set_ylim(0, 1.05)
ax.set_title("QAOA quality across depths")
ax.legend(loc="best")

# Annotate values on each bar.
for bar, val in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", fontsize=9)
for bar, val in zip(bars2, gs_probs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 9. Persist outputs

For the polish phase (Weeks 7–8) we'll want these QAOA results to plot alongside hardware runs from Week 5. Save the converged parameters, traces, counts, and summary metrics.

In [ ]:
qaoa_outputs = {
    "shots":            SHOTS,
    "seed":             SEED,
    "ising_offset":     ISING_OFFSET,
    "ground_energy":    E_GROUND,
    "ground_bitstring": GROUND_BITS,
}
for p in [1, 2, 3]:
    res = results[p]
    qaoa_outputs[f"p{p}_params"]   = res.optimal_params
    qaoa_outputs[f"p{p}_energy"]   = res.optimal_energy
    qaoa_outputs[f"p{p}_trace"]    = res.trace
    qaoa_outputs[f"p{p}_ratio"]    = summary[p]["ratio"]
    qaoa_outputs[f"p{p}_gs_prob"]  = summary[p]["gs_prob"]
    qaoa_outputs[f"p{p}_best_E"]   = summary[p]["best_sampled"]
    # Counts as parallel arrays (npz can't save dicts directly).
    qaoa_outputs[f"p{p}_count_bitstrings"] = np.array(list(res.counts.keys()))
    qaoa_outputs[f"p{p}_count_values"]     = np.array(list(res.counts.values()))

np.savez(DATA_PROCESSED / "qaoa_simulator_results.npz", **qaoa_outputs)
print(f"Saved {(DATA_PROCESSED / 'qaoa_simulator_results.npz').relative_to(ROOT)}")
print(f"  {len(qaoa_outputs)} entries across p = 1, 2, 3.")

## Session 3 — wrap-up

We've now run QAOA end-to-end on the same portfolio Hamiltonian whose ground state was identified classically in Session 2, on an exact statevector simulator. Concretely:

- Built a $p$-layer parameterised ansatz with $2p$ real parameters using `PauliEvolutionGate` for the cost layer and explicit $R_X$ rotations for the mixer.
- Optimised those parameters with COBYLA against the exact expectation value $\langle\psi|\hat H|\psi\rangle$.
- Sampled the optimised state and translated each bitstring back to a portfolio selection.
- Reported approximation ratio, ground-state probability, and best sampled energy per depth.

The approximation ratio behaviour gives us a baseline for what "noiseless QAOA" looks like on this Hamiltonian. When we move to noisy simulation (Week 4) and real hardware (Week 5), we'll plot those runs against the curves saved here to quantify the gap.

**Next session (Session 4):** examine how QAOA's behaviour depends on the choice of risk-aversion parameter $q$ and budget $B$. Specifically we want to see whether there's a "sweet spot" where the Ising landscape has a healthy spectral gap and QAOA's ground-state probability rises, versus regimes where the optimal portfolio is uninteresting (e.g. just-pick-highest-returns or just-pick-lowest-vols).